# Reproducibility Notebook

This notebook reproduces the main results of the strict no-detuning multitone-SQR study. It is designed for both users and future agents: the early parameter cell collects the main knobs, the rerun cell can regenerate the saved artifacts, and the remaining cells load the archived results to reproduce the headline tables and figures.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# User-tunable parameters
STUDY_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'scripts' else Path(r'C:\\Users\\jl82323\\Box\\Shyam Shankar Quantum Circuits Group\\Users\\Users_JianJun\\cqed_based_study\\studies\\multitone_sqr_no_go_decoupled_echo_dispersive_cqed')
DATA_DIR = STUDY_DIR / 'data'
FIGURES_DIR = STUDY_DIR / 'figures'
SCRIPTS_DIR = STUDY_DIR / 'scripts'
RERUN_MAIN = False
RERUN_VALIDATION = False
RUN_STUDY_ARGS = ['--n-starts', '1', '--maxiter', '5']
SHOW_FIGURES = True


## Optional rerun

Set `RERUN_MAIN` and/or `RERUN_VALIDATION` to `True` in the parameter cell above if you want to regenerate the archived results instead of only loading them.

In [ ]:
if RERUN_MAIN:
    subprocess.run([sys.executable, str(SCRIPTS_DIR / 'run_study.py'), *RUN_STUDY_ARGS], check=True, cwd=STUDY_DIR)
if RERUN_VALIDATION:
    subprocess.run([sys.executable, str(SCRIPTS_DIR / 'validate_results.py')], check=True, cwd=STUDY_DIR)
print('Rerun step complete.')


## Load archived artifacts

In [ ]:
def load_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

results = load_json(DATA_DIR / 'study_results.json')
summary = load_json(DATA_DIR / 'study_summary.json')
analytic = load_json(DATA_DIR / 'analytic_summary.json')
validation = load_json(DATA_DIR / 'validation_summary.json')
validation_details = load_json(DATA_DIR / 'validation_details.json')
audit = load_json(DATA_DIR / 'prior_audit.json')
print('Loaded rows:', len(results['case_rows']))
print('Runtime [s]:', results['runtime_s'])


## Headline findings

In [ ]:
headline = {
    'strict_shared_line_mean_fidelity': summary['strict_shared_line_mean_fidelity'],
    'strict_shared_line_best_fidelity': summary['strict_shared_line_best_fidelity'],
    'strict_shared_line_mean_max_residual_z_error_rad': summary['strict_shared_line_mean_max_residual_z_error_rad'],
    'decoupled_block_min_fidelity': summary['decoupled_block_min_fidelity'],
    'ideal_echo_mean_fidelity': summary['ideal_echo_mean_fidelity'],
    'finite_echo_mean_fidelity': summary['finite_echo_mean_fidelity'],
}
headline


## Tabulate the saved case rows

In [ ]:
import pandas as pd

df = pd.DataFrame(results['case_rows'])
full = df[df['construction'] == 'full_shared_line']
display(full.groupby('family')['restricted_average_gate_fidelity'].agg(['mean', 'min', 'max']).round(6))
display(full.groupby('n_active')['restricted_average_gate_fidelity'].agg(['mean', 'min', 'max']).round(6))
display(df.groupby('construction')['restricted_average_gate_fidelity'].agg(['mean', 'min', 'max']).round(6))


## Validation details

In [ ]:
validation_details


## Prior-work audit

In [ ]:
audit


## Main figures

In [ ]:
if SHOW_FIGURES:
    from IPython.display import Image, display
    for name in [
        'duration_fidelity_tradeoff.png',
        'blockwise_residual_z_vs_duration.png',
        'addressed_subspace_scaling.png',
        'plain_vs_echo_comparison.png',
    ]:
        path = FIGURES_DIR / name
        print(path.name)
        display(Image(filename=str(path)))


## Next steps

If you want to extend the study, the most natural follow-ups are:
1. Replace the strict shared-line square multitone ansatz with a richer segmented or sampled envelope while still forbidding artificial per-tone detuning.
2. Test whether selective or manifold-adaptive refocusing pulses can outperform the strict echoed construction studied here.
3. Extend the same no-go analysis to a model with explicit open-system noise or additional hardware filtering.